本教程介绍如何将UniParser解析结果与第三方库(`PyMuPDF`, `Pillow`)结合，实现PDF中各类图片的精准提取和裁剪。

我们将直接解析 `pages_tree` 格式数据，展示如何利用返回的 `bbox`（边界框）在本地PDF渲染画面上将图片扣取下来。通过这个例子，你也能更好地理解 UniParser 解析数据的层级结构。

## 导入依赖
> 你可能需要提前安装 `PyMuPDF` 和 `Pillow`:
> `pip install PyMuPDF Pillow requests`

In [1]:
import json
import os
import time
import requests
from collections import Counter
from pathlib import Path

import fitz  # PyMuPDF
from PIL import Image

from uniparser_tools.api.clients import UniParserClient
from uniparser_tools.common.constant import FormatFlag, ParseMode, ParseModeTextual

## 初始化

In [ ]:
host = "https://uniparser.dp.tech/"  # 官网

# 替换为你的认证 api key
api_key = "xxxxxxx"
# 初始化客户端
parser = UniParserClient(host=host, api_key=api_key)

# 创建一个目录来保存切图结果
save_dir = "./outputs/crop_images"
os.makedirs(save_dir, exist_ok=True)
print("✅ 准备就绪")

✅ 准备就绪


## 1. 提交解析任务并获取 JSON 树结构 (Pages Tree)
为了能获取每个元素的 `bbox` (包含x1, y1, x2, y2的相对坐标)，我们需要使用带有丰富层级信息的 `pages_tree` 格式数据。

In [5]:
# 设置解析文件路径
pdf_path = "./tasks/He_Deep_Residual_Learning_CVPR_2016_paper.pdf"

# 提交解析任务
trigger_file_result = parser.trigger_file(
    pdf_path,
    textual=ParseModeTextual.DigitalExported,
    table=ParseMode.OCRFast,
    molecule=ParseMode.OCRFast,
    chart=ParseMode.DumpBase64,
    figure=ParseMode.DumpBase64,
    expression=ParseMode.DumpBase64,
    equation=ParseMode.OCRFast,
)

if trigger_file_result["status"] != "success":
    print(json.dumps(trigger_file_result, indent=4))
    raise Exception("trigger file failed")

token = trigger_file_result['token']
print(f"任务提交成功, Token: {token}")

任务提交成功, Token: df84e1c227415776be33b7e0c97c9baa


In [6]:
# 等待并获取 pages_tree 数据
json_data = None
print("正在获取 pages_tree 数据...")
for i in range(5):
    result = parser.get_formatted(
        token,
        content=False,
        objects=False,
        pages_dict=False,
        pages_tree=True,
        molecule_source=False,
        textual=FormatFlag.Plain,
    )
    
    if result.get("status") == "success" and result.get("pages_tree") is not None:
        json_data = result["pages_tree"]
        print(f"✅ 获取成功，共 {len(json_data)} 页数据")
        break
    
    print(f"⏳ 数据尚未准备好，等待重试... ({i+1}/5)")
    time.sleep(3)

if not json_data:
    raise Exception("获取 pages_tree 数据失败")

正在获取 pages_tree 数据...
✅ 获取成功，共 9 页数据


## 2. 理解并遍历 Pages Tree 数据
`pages_tree` 是一棵复杂的嵌套树。我们通过递归函数 `traverse_tree` 把它展开为一维列表，提取其中具有 `bbox` 的对象。

支持的图片相关类型包括: `figure`, `image`, `figuregroup`, `chart`, `table`, `formula`, `equation`。

In [7]:
def traverse_tree(node, result_list):
    """递归遍历 JSON 节点树，提取所有包含 bbox 和 type 的节点"""
    if isinstance(node, list):
        for item in node:
            traverse_tree(item, result_list)
        return
    
    if isinstance(node, dict):
        if 'type' in node and 'bbox' in node:
            result_list.append(node)
        
        if 'items' in node and isinstance(node['items'], list):
            traverse_tree(node['items'], result_list)

# 提取全局所有的可裁剪对象节点
all_nodes = []
traverse_tree(json_data, all_nodes)

print(f"全文共找到 {len(all_nodes)} 个具有边界框的结构化元素")

全文共找到 222 个具有边界框的结构化元素


In [8]:
# 看看这篇文章里都有哪些图片元素
IMAGE_TYPES = ['figure', 'image', 'figuregroup', 'chart', 'table', 'formula', 'equation']

type_counter = Counter([node.get('type', 'Unknown') for node in all_nodes])
image_types_counts = {k: v for k, v in type_counter.items() if k in IMAGE_TYPES}

print("📊 PDF 图片类型元素统计:")
for img_type, count in sorted(image_types_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  - {img_type:15s}: {count:3d} 个")

📊 PDF 图片类型元素统计:
  - chart          :   9 个
  - table          :   9 个
  - image          :   7 个
  - figure         :   4 个
  - equation       :   2 个


## 3. 核心切图逻辑
下面我们将使用 `PyMuPDF` 和 `Pillow`。
核心原理：
1. **计算相对坐标**: UniParser返回的 `bbox` 值（x1, y1, x2, y2）是在 `[0, 1]` 之间的**相对坐标**。
2. **渲染页面**: 将PDF单页按照给定的 DPI（如150）渲染为位图。
3. **还原绝对坐标并裁剪**: 将相对坐标乘以图片的宽高，得到绝对像素坐标，然后使用 `Pillow.crop` 抠出图像。

In [9]:
def crop_pdf_elements(pdf_path, json_data, output_dir, target_types=None, dpi=150):
    """
    核心切图函数
    """
    if target_types is None:
        target_types = IMAGE_TYPES
        
    # 使用 PyMuPDF 打开原始文档
    doc = fitz.open(pdf_path)
    
    # PDF的默认DPI通常为72，我们通过 scale_factor 放大渲染倍数来提升清晰度
    scale_factor = dpi / 72.0
    
    saved_files = []
    print(f" 开始切图 (DPI: {dpi}) | 目标类型: {target_types} ...")
    
    # 遍历每一页
    for page_idx in range(len(doc)):
        page = doc[page_idx]
        page_data = json_data[page_idx] if isinstance(json_data, list) else json_data
        
        # 1. 展开当前页的节点
        page_nodes = []
        traverse_tree(page_data, page_nodes)
        
        # 过滤出我们需要的目标类型
        target_nodes = [n for n in page_nodes if n.get('type') in target_types]
        if not target_nodes:
            continue
            
        # 2. 将当前PDF页渲染为图片
        mat = fitz.Matrix(scale_factor, scale_factor)
        pix = page.get_pixmap(matrix=mat)
        mode = "RGBA" if pix.alpha else "RGB"
        page_image = Image.frombytes(mode, [pix.width, pix.height], pix.samples)
        
        # 页面真实宽高（注意这和渲染出来的pix宽高的比例就是 scale_factor）
        page_w, page_h = page.rect.width, page.rect.height
        
        # 3. 遍历节点进行裁剪
        for idx, node in enumerate(target_nodes):
            node_type = node.get('type')
            bbox = node.get('bbox')
            if not bbox:
                continue
                
            # bbox格式：{'x1': 0.1, 'y1': 0.1, 'x2': 0.5, 'y2': 0.5}
            # 将相对坐标转换为渲染图片的实际像素坐标
            x1 = int(bbox['x1'] * page_w * scale_factor)
            y1 = int(bbox['y1'] * page_h * scale_factor)
            x2 = int(bbox['x2'] * page_w * scale_factor)
            y2 = int(bbox['y2'] * page_h * scale_factor)
            
            # 防御性判断坐标是否有效
            if x2 <= x1 or y2 <= y1:
                continue
                
            # 在整页图片上执行裁剪
            cropped = page_image.crop((x1, y1, x2, y2))
            
            # 保存文件
            filename = f"page{page_idx:03d}_{node_type}_{idx:03d}.png"
            filepath = os.path.join(output_dir, filename)
            cropped.save(filepath, 'PNG')
            saved_files.append(filepath)
            
        print(f"✅ 第 {page_idx} 页: 裁剪了 {len(target_nodes)} 张图片")
        
    doc.close()
    print(f"🎉 切图完成！共保存在 {output_dir} 下的 {len(saved_files)} 个文件中。\n")
    return saved_files

### 示例 1：提取所有的图表和表格
我们可以只提取 `figure` 和 `table`。

In [10]:
if os.path.exists(pdf_path):
    out_path1 = os.path.join(save_dir, "figs_and_tables")
    os.makedirs(out_path1, exist_ok=True)
    
    # 裁剪 DPI 我们设为 150，对于一般屏幕展示足够清晰
    cropped_files_1 = crop_pdf_elements(
        pdf_path=pdf_path, 
        json_data=json_data, 
        output_dir=out_path1, 
        target_types=['figure', 'table'],
        dpi=150
    )

 开始切图 (DPI: 150) | 目标类型: ['figure', 'table'] ...
✅ 第 1 页: 裁剪了 1 张图片
✅ 第 3 页: 裁剪了 1 张图片
✅ 第 4 页: 裁剪了 2 张图片
✅ 第 5 页: 裁剪了 5 张图片
✅ 第 6 页: 裁剪了 2 张图片
✅ 第 7 页: 裁剪了 2 张图片
🎉 切图完成！共保存在 ./outputs/crop_images/figs_and_tables 下的 13 个文件中。



### 示例 2：高精度提取所有公式类型
对于 `equation`（独立公式）我们往往需要更高的分辨率，比如 `DPI=300`。

In [11]:
if os.path.exists(pdf_path):
    out_path2 = os.path.join(save_dir, "equations_highres")
    os.makedirs(out_path2, exist_ok=True)
    
    # 使用 300 DPI
    cropped_files_2 = crop_pdf_elements(
        pdf_path=pdf_path, 
        json_data=json_data, 
        output_dir=out_path2, 
        target_types=['equation', 'formula'],
        dpi=300
    )

 开始切图 (DPI: 300) | 目标类型: ['equation', 'formula'] ...
✅ 第 2 页: 裁剪了 2 张图片
🎉 切图完成！共保存在 ./outputs/crop_images/equations_highres 下的 2 个文件中。



## 总结

通过上面这个完整的脚本流，你可以看到如何将 UniParser 接口返回的高度结构化的 `pages_tree`，与本地 Python 图像处理库联动。  
